In [ ]:
import os
import datetime
import colormaps
from pathlib import Path
from functools import partial
from itertools import product
from string import ascii_lowercase
from tqdm import tqdm, trange
import matplotlib.pyplot as plt
from matplotlib.dates import DateFormatter
from matplotlib.colors import (
    LinearSegmentedColormap,
    BoundaryNorm,
    Normalize,
    rgb_to_hsv,
    hsv_to_rgb,
)
from matplotlib.ticker import MaxNLocator
import numpy as np
import pandas as pd
import xarray as xr
import polars as pl
import polars.selectors as cs

os.environ["PATH"] = os.environ["PATH"] + ":/opt/homebrew/bin"

from jetutils.definitions import (
    PRETTIER_VARNAME,
    YEARS,
    UNITS,
    RESULTS,
    FACTORS_UNITS,
    FACTORS,
    DATADIR,
    DUNCANS_REGIONS_NAMES,
    MONTH_NAMES,
    FIGURES,
    RADIUS,
    get_region,
    squarify,
    polars_to_xarray,
    xarray_to_polars,
)
from jetutils.data import standardize, standardize_polars_dtypes, compute_anomalies_pl
from jetutils.geospatial import (
    central_diff,
    haversine_from_dl,
    compute_relative_clim,
    compute_relative_sm,
    compute_relative_std,
    compute_relative_anom,
    create_all_relative_plots,
)
from jetutils.jet_finding import (
    average_jet_categories,
    track_jets,
    pers_from_cross,
    spells_from_cross_catd_simple,
    extract_features,
    find_all_jets,
    jet_position_as_da,
    to_one_large,
)
from jetutils.plots import (
    STYLE_SHEET,
    COLORS,
    COLORS_EXT,
    WERNLI_FLAIR,
    WERNLI_FLAIR_LEVELS,
    Clusterplot,
    plot_interp,
    plot_relative_time,
    gaussian_kde,
)
from jetutils.anyspell import (
    make_daily,
    mask_from_spells_pl,
    subset_around_onset,
    extend_spells,
    get_spells,
)
from jetutils.stats import create_bootstrapped_times, odds_ratio, is_signif_OR, common_OR

%load_ext IPython.extensions.autoreload
%autoreload 2
%matplotlib inline
%config InlineBackend.print_figure_kwargs = {'bbox_inches':None}

In [ ]:
both_props = {}
both_phat_props = {}
for run in ["historical", "ssp370"]:
    basepath = Path(DATADIR, run)
    jets = pl.read_parquet(basepath.joinpath("jets.parquet"))
    both_props[run] = pl.read_parquet(basepath.joinpath("props.parquet"))
    both_phat_props[run] = pl.read_parquet(basepath.joinpath("props_full.parquet"))
    cross_catd = pl.read_parquet(basepath.joinpath("cross.parquet"))
    break

In [ ]:
jets_ = jets.filter(pl.col("time").dt.month() == 7, pl.col("time").dt.ordinal_day() < 190)

In [ ]:
from jetutils.jet_finding import is_polar_gmix
jets_newcat = is_polar_gmix(jets, ("s", "theta"), mode="week", n_init=3, init_params="kmeans", v2=True)

In [ ]:
xys = []
xys = np.array(xys)
fig, axes = plt.subplots(
    3, 4, figsize=(11, 9), constrained_layout=True, sharex="all", sharey="all"
)
axes = axes.ravel()
pair = ["s", "theta", "is_polar"]
cmap = LinearSegmentedColormap.from_list(
    "pinkredpurple", [COLORS[2], COLORS_EXT[8], COLORS[1]]
)
bins = np.linspace(0, 1, 31)
gridsize = 18
for month in range(1, 13):
    ax = axes[month - 1]
    X = extract_features(jets_newcat, pair, season=month)
    probas = X[pair[2]]
    center_stj = X.filter(pl.col("is_polar") < 0.3).mean()
    center_edj = X.filter(pl.col("is_polar") > 0.7).mean()
    X1D = X["is_polar"]

    im1 = ax.hexbin(X[pair[0]], X[pair[1]], cmap=colormaps.gray_r, gridsize=gridsize)
    im2 = ax.hexbin(
        X[pair[0]], X[pair[1]], C=probas, cmap=colormaps.gray_r, gridsize=gridsize
    )

    plt.draw()

    offsets1 = np.asarray(list(map(tuple, im1.get_offsets())), dtype="f, f")
    offsets2 = np.asarray(list(map(tuple, im2.get_offsets())), dtype="f, f")
    mask12 = np.isin(offsets1, offsets2)
    colors = cmap(im2.get_array())
    colors = rgb_to_hsv(colors[:, :3])
    min_s, max_s = 0.0, 0.9
    min_v, max_v = 0.9, 1.0
    scaling = np.clip(
        im1.get_array()[mask12] / im1.get_array()[mask12].max() * 1.1, 0, 1
    )
    f = lambda x: np.pow(x, 2)
    colors[:, 1] = min_s + f(scaling ** 1.5) * (max_s - min_s)
    colors[:, 2] = max_v - f(scaling) * (max_v - min_v)
    colors = hsv_to_rgb(colors)
    im2.set_array(None)
    im2.set_facecolor(colors)
    # im2.set_linewidths(0.2)
    im2.set_linewidths(np.clip(2 - 3 * (scaling), 0, 2))
    im2.set_edgecolor("white")
    im2 = ax.add_collection(im2)
    if month > 8:
        label = PRETTIER_VARNAME.get(pair[0], pair[0])
        unit = UNITS.get(pair[0], "$~$")
        ax.set_xlabel(f"{label} [{unit}]")
    if month % 4 == 1:
        label = PRETTIER_VARNAME.get(pair[1], pair[1])
        unit = UNITS.get(pair[1], "$~$")
        ax.set_ylabel(f"{label} [{unit}]")
    if pair[0] in ["lev", "theta"]:
        ax.invert_xaxis()
    elif pair[1] in ["lev", "theta"]:
        ax.invert_yaxis()

    ax.set_title(MONTH_NAMES[month - 1])
    ax.scatter(
        *pl.concat([center_stj, center_edj])[pair[:2]].to_numpy().T,
        facecolor="black",
        edgecolor="white",
        marker="X",
        linewidths=1,
        s=45,
    )
    iax = ax.inset_axes([0.65, 0.14, 0.5, 0.5])
    X1D = np.clip(X1D, 0, 1)
    iax.hist(X1D, bins=bins, alpha=0.5, color="black")
    iax.hist(X1D[probas > 0.5], bins=bins, alpha=1.0, color=COLORS[1])
    iax.hist(X1D[probas < 0.5], bins=bins, alpha=1.0, color=COLORS[2])
    iax.set_xticks([0, 0.5, 1])
    iax.set_yticks([])
    iax.spines[["left", "right", "top"]].set_visible(False)
    iax.set_facecolor((1.0, 1.0, 1.0, 0.0))
    plt.draw()
    # break
# fig.savefig(f"{FIGURES}/FeatureBased/gmix_demo.pdf")

In [ ]:
from matplotlib.dates import MonthLocator
from jetutils.data import periodic_rolling_pl
from matplotlib.lines import Line2D
# data_vars: list = ["mean_lat", "mean_s", "mean_theta", "tilt", "wavinessR16", "width", "pers", "int", "int_over_europe"]
data_vars: list = ["mean_lat", "mean_s", "tilt", "wavinessR16", "pers", "int_over_europe"]
nrows: int = 2
ncols: int = 3

plot_kwargs = {"historical": [both_phat_props["historical"], "solid"], "ssp370": [both_phat_props["ssp370"], "dashed"]}

fig, axes = plt.subplots(
    nrows,
    ncols,
    figsize=(ncols * 3.5, nrows * 2.4),
    constrained_layout=True,
    sharex="all",
)
axes = axes.flatten()
jets = both_phat_props["historical"]["jet"].unique().to_numpy()
njets = len(jets)
gb_args = [pl.col("time").dt.ordinal_day().alias("dayofyear"), pl.col("jet")]
aggs = [
    pl.col(col).replace([float("-inf"), float("inf"), float("nan")], None).mean()
    for col in data_vars
]


for name, args in plot_kwargs.items():
    props, ls = args
    gb = props.group_by(gb_args)

    means = gb.agg(aggs).sort("dayofyear", "jet", descending=[False, True])
    means = squarify(means, ["dayofyear", "jet"])
    means = periodic_rolling_pl(means, 15, data_vars)

    x = means["dayofyear"].unique()
    color_order = [2, 1]
    for letter, varname, ax in zip(ascii_lowercase, data_vars, axes.ravel()):
        dji = varname == "double_jet_index"
        factor = FACTORS.get(varname, 1)
        if factor == 1:
            factor_str = ""
        else:
            factor_str = str(int(np.log10(np.abs(factor))))
            factor_str = r"$10^{" + factor_str + r"} \times $"
        ys = means[varname].to_numpy().reshape(366, njets)
        for i in range(njets):
            color = "black" if dji else COLORS[color_order[i]]
            ax.plot(
                x,
                ys[:, i] / factor,
                lw=3,
                color=color,
                zorder=10,
                ls=ls,
            )
            if dji:
                break
        ax.set_title(
            f"{letter}) {PRETTIER_VARNAME.get(varname, varname)} [{factor_str}{UNITS.get(varname, '')}]"
        )
        ax.xaxis.set_major_locator(MonthLocator(range(0, 13, 3)))
        ax.xaxis.set_major_formatter(DateFormatter("%b"))
        ax.set_xlim(min(x), max(x))
        if varname == "mean_lev":
            ax.invert_yaxis()
        # if name == "dobl":
        #     ylim = ax.get_ylim()
        #     wherex = np.isin(x, JJASDOYS)
        #     ax.fill_between(
        #         x, *ylim, where=wherex, alpha=0.1, color="black", zorder=-10
        #     )
        #     ax.set_ylim(ylim)
           
# linedata =  [[0, 1], [0, 1]]
# handles = [
#     Line2D(*linedata, lw=2, ls="solid", color=COLORS[2]),
#     Line2D(*linedata, lw=2, ls="solid", color=COLORS[1]),
#     Line2D(*linedata, lw=2, ls="solid", color="black"),
#     Line2D(*linedata, lw=2, ls="dashed", color="black"),
# ]
# labels = ["STJ", "EDJ", "ctrl", "dobl"]
# axes.ravel()[1].legend(ncol=2, handles=handles, labels=labels, labelspacing=0.2, handletextpad=0.2, columnspacing=0.01, handlelength=1.1).set_zorder(102)
# fig.savefig(f"{FIGURES}/Diabatic/seasonal.pdf")